# Phase-A.0 Rev-3.4 — Notebook 0B: On-chain × BanRep Bridge-Validation (Colombia)

**Task 11.C** of `contracts/docs/superpowers/plans/2026-04-20-remittance-surprise-implementation.md` (Rev-3.4).

**Purpose.** Cross-validate that the daily on-chain COPM + cCOP flow aggregate (Task 11.A) — summed by the Task-11.B Friday-anchored weekly aggregator and re-aggregated to calendar quarters — correlates with the Task-11 BanRep quarterly remittance series (`value_usd`, series 4150). This is the Rev-3 bridge between the daily-native primary X and the legacy quarterly BanRep signal: the result shapes the economic interpretation of the Phase-A.0 regression but does NOT gate whether the primary regression runs (the on-chain X is a well-defined observable either way).

**Anti-fishing framing (Rev-1 §10 / Rev-3 plan line 308).** This exercise is a distinct pre-commitment on the remittance external-inflow channel via the on-chain COPM+cCOP rail. It is **NOT a rescue** of the CPI-FAIL verdict (2026-04-19). Provenance anchor: `/home/jmsbpp/apps/liq-soldk-dev/notes/REMITTANCE_VOLATILITY_SWAP/REMITTANCE_VOLATILITY_SWAP.md` has mtime **2026-04-02**, predating the CPI-FAIL verdict by 17 days.

**Inputs.**
1. `contracts/data/copm_ccop_daily_flow.csv` — Task-11.A 585-row daily fixture (2024-09-17 → 2026-04-24, Task-11.A 8-column schema).
2. `contracts/scripts/weekly_onchain_flow_vector.py::aggregate_daily_to_weekly_vector` — Task-11.B pure aggregator, Friday-anchored weeks (America/Bogota), 6-channel weekly vector.
3. `contracts/data/banrep_remittance_aggregate_monthly.csv` — Task-11 BanRep aggregate series (filename retained from Rev-1 spec; contents are QUARTERLY per Rev-3 escalation, 104 rows 2000-Q1 → 2025-Q4).

**Overlap window — observed N = 6 quarters (off-by-one from task prescription N = 7).** Quarter-end rows present in BanRep CSV for the overlap window (Task-11.A first-obs 2024-09-17 falls in 2024-Q3; BanRep last-obs 2025-12-31 in 2025-Q4):

```
2024-09-30  (2024-Q3)
2024-12-31  (2024-Q4)
2025-03-31  (2025-Q1)
2025-06-30  (2025-Q2)
2025-09-30  (2025-Q3)
2025-12-31  (2025-Q4)
```

Direct enumeration gives six quarter-end rows, yielding **N = 6** observations for the Pearson ρ and **5** Δ Q-over-Q transitions for sign-concordance. The task description stated N=7 (plausibly including 2024-Q3 partially as a contributor and counting transitions rather than observations); the notebook and scratch log record the actual observed N=6. This discrepancy is flagged here (in the pre-commitment header), resolved before §3's ρ computation, and does NOT modify the pre-registered gate thresholds.

## Anti-fishing: Pre-registered Bridge Gate (Rev-3 plan line 316-319)

The following gate logic is committed **BEFORE** any correlation is computed. The notebook-author MUST NOT adjust the thresholds after observing ρ (this is the core anti-fishing discipline inherited from the CPI-surprise exercise).

| Verdict | Condition |
|---|---|
| **PASS-BRIDGE** | ρ > 0.5 AND sign-concordant on Δ quarter-over-quarter (strict majority — at least ⌈(K+1)/2⌉ of K transitions agree; K=5 ⇒ ≥3 of 5) |
| **INCONCLUSIVE-BRIDGE** | 0.3 < ρ ≤ 0.5 AND sign-concordant |
| **FAIL-BRIDGE** | ρ ≤ 0.3 OR sign-discordant (strict-minority concordance count) |

Concrete rule at observed K = 5 transitions: sign-concordant ≡ (# concordant Δ-sign transitions) ≥ 3. This is committed BEFORE computing ρ and is a function of the number of transitions, so it is stable under off-by-one on N.

## NaN-ambiguity rule (Rev-3.4 addition — Task-11.B code-review subtlety #1)

The Task-11.B aggregator emits NaN on a weekly row for two economically distinct reasons:
1. **partial_week** — fewer than 7 daily observations contributed (boundary weeks at the start/end of the fixture). This is a no-data condition and MUST be dropped from quarterly aggregation.
2. **all_zero_full_week** — 7 daily observations present but all four daily USD channels (`copm_mint_usd`, `copm_burn_usd`, `ccop_usdt_inflow_usd`, `ccop_usdt_outflow_usd`) are zero for every day. The concentration-HHI denominator is zero in this case and the aggregator emits NaN for `flow_concentration_w`, which then propagates to the NaN-check. This is a valid zero-flow observation and MUST be included in the quarterly sum at value 0 (its `flow_sum_w` is zero whether reported as NaN or as 0).

§1 of this notebook annotates each weekly row with a `nan_reason_w` enum in `{"partial_week", "all_zero_full_week", "valid"}` using the underlying daily data (a rows-per-week count + sum-of-absolute daily net flow from the Task-11.A CSV). §2 uses `nan_reason_w` to drop `partial_week` rows and treat `all_zero_full_week` rows as valid zeros at quarterly aggregation.

## Gate Verdict

*populated by §4 code cell; emitted to `contracts/.scratch/2026-04-20-onchain-banrep-bridge-result.md` as the final artifact of this notebook.*

## §1 — Data alignment onto a common quarterly index

**Reference.** Task-11.A daily fixture (`contracts/data/copm_ccop_daily_flow.csv`, 585 rows, 2024-09-17 → 2026-04-24, 8-column schema). Task-11.B aggregator (`contracts/scripts/weekly_onchain_flow_vector.py::aggregate_daily_to_weekly_vector`, Friday-anchored weekly resampling). Task-11 BanRep CSV (`contracts/data/banrep_remittance_aggregate_monthly.csv`, 104 quarterly rows ending 2025-12-31, header column `reference_period,value_usd,mpr_vintage_date,source_url`).

**Why this step is used.** The three sources live on three distinct time grids (daily calendar dates, Friday-anchored weekly right-labels, quarter-end-date quarterly). The bridge-ρ requires a single common grid with one observation per quarter for each series. The cleanest shared grid is the calendar quarter-end boundary. The overlap is six quarter-end boundaries (2024-09-30, 2024-12-31, 2025-03-31, 2025-06-30, 2025-09-30, 2025-12-31), with 2024-Q3 as the first quarter (Task-11.A starts 2024-09-17, mid-Q3) and 2025-Q4 as the last (BanRep ends 2025-Q4). That gives **N = 6** overlap quarters and **K = 5** Q-over-Q transitions.

**Relevance to results.** The overlap window size directly caps the statistical power of the bridge-ρ: N=6 gives a Pearson CI roughly ±0.7 wide (Fisher z-transform, small-sample approximation). The pre-registered gate thresholds (0.3 / 0.5) were chosen with this N in mind — intermediate values are classified INCONCLUSIVE precisely because 6 observations cannot distinguish a genuine 0.4 ρ from a noisy 0.7 ρ. The `nan_reason_w` annotation is required here (not at §2 time) because the quarterly-aggregation step needs to see each weekly row's disposition before summing.

**Connection to the primary regression.** The primary regression operates on the weekly 6-channel vector directly; the quarterly bridge is an economic-narrative validation only. FAIL-BRIDGE shifts the interpretation from "remittance" to "crypto-rail income-conversion" but does NOT halt the primary regression (Rev-3.1 recovery protocol item 1).

In [1]:
"""§1 — Data alignment.

Loads the three inputs, computes the Task-11.B weekly vector, annotates each
weekly row with a `nan_reason_w` enum per the Rev-3.4 rule, and emits an
alignment-summary DataFrame keyed on quarter-end.

Zero network I/O: all three sources are local CSVs or local Python modules.
Deterministic given the fixed input files.
"""
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd

# ── Path setup ────────────────────────────────────────────────────────────
# Notebook cwd is the directory containing 0B_bridge_validation.ipynb (or
# the worktree root when executed by nbconvert from a shell). Resolve
# relative to the worktree root using the env_remittance helper.
_THIS_DIR = Path.cwd()
# Walk up looking for the worktree root (heuristic: contains contracts/).
_root = _THIS_DIR
while _root.parent != _root and not (_root / "contracts").is_dir():
    _root = _root.parent
if not (_root / "contracts").is_dir():
    # nbconvert from worktree-root cwd: _THIS_DIR already is the root.
    _root = _THIS_DIR
WORKTREE_ROOT = _root

DAILY_CSV_PATH = WORKTREE_ROOT / "contracts" / "data" / "copm_ccop_daily_flow.csv"
BANREP_CSV_PATH = (
    WORKTREE_ROOT / "contracts" / "data" / "banrep_remittance_aggregate_monthly.csv"
)

# Make the aggregator importable without forcing the user to run pytest-style.
_SCRIPTS_DIR = WORKTREE_ROOT / "contracts" / "scripts"
if str(_SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(_SCRIPTS_DIR))

from weekly_onchain_flow_vector import (  # noqa: E402 (sys.path extended above)
    WEEKLY_VECTOR_COLUMNS,
    aggregate_daily_to_weekly_vector,
)

# ── Load daily on-chain CSV (Task-11.A) ───────────────────────────────────
daily = pd.read_csv(DAILY_CSV_PATH, comment="#", parse_dates=["date"])
print(f"[§1] Loaded daily CSV: {len(daily)} rows "
      f"({daily['date'].min().date()} → {daily['date'].max().date()})")

# ── Load BanRep quarterly CSV (Task-11) ───────────────────────────────────
banrep = pd.read_csv(
    BANREP_CSV_PATH,
    comment="#",
    parse_dates=["reference_period", "mpr_vintage_date"],
)
print(f"[§1] Loaded BanRep quarterly CSV: {len(banrep)} rows "
      f"({banrep['reference_period'].min().date()} → "
      f"{banrep['reference_period'].max().date()})")

# ── Compute Task-11.B weekly vector (Friday-anchored) ─────────────────────
weekly = aggregate_daily_to_weekly_vector(daily)
print(f"[§1] Task-11.B weekly vector: {len(weekly)} rows "
      f"({weekly.index.min().date()} → {weekly.index.max().date()})")
print(f"[§1] Weekly vector columns: {list(weekly.columns)}")

# ── NaN-reason annotation (Rev-3.4 addition) ──────────────────────────────
# We recompute the per-week daily count + sum-of-abs from the daily CSV so
# we can disambiguate the two NaN sources the aggregator produces:
#   * partial_week      — fewer than 7 daily rows contributed.
#   * all_zero_full_week — 7 rows present, all four USD channels zero.
#   * valid             — otherwise (weekly row has a meaningful value).
daily_indexed = daily.set_index("date").sort_index()
net_flow_abs = (
    daily_indexed["copm_mint_usd"].fillna(0.0)
    + daily_indexed["copm_burn_usd"].fillna(0.0)
    + daily_indexed["ccop_usdt_inflow_usd"].fillna(0.0)
    + daily_indexed["ccop_usdt_outflow_usd"].fillna(0.0)
).abs()
weekly_count = net_flow_abs.resample("W-FRI", label="right", closed="right").size()
weekly_abs_sum = net_flow_abs.resample("W-FRI", label="right", closed="right").sum()


def _classify_nan_reason(count: int, abs_sum: float) -> str:
    """Enum in {'partial_week', 'all_zero_full_week', 'valid'}."""
    if count < 7:
        return "partial_week"
    # count == 7: either all zero (denominator=0 → aggregator NaN on HHI
    # but flow_sum_w itself is 0) or truly valid with positive flow.
    if abs_sum == 0.0:
        return "all_zero_full_week"
    return "valid"


# Reindex weekly_count / weekly_abs_sum onto the weekly vector's index.
# They should already be aligned by construction (same resample kwargs).
weekly = weekly.copy()
weekly["_daily_count"] = weekly_count.reindex(weekly.index).astype("Int64")
weekly["_daily_abs_sum"] = weekly_abs_sum.reindex(weekly.index).astype("float64")
weekly["nan_reason_w"] = [
    _classify_nan_reason(int(c), float(s))
    for c, s in zip(
        weekly["_daily_count"].fillna(0).astype(int),
        weekly["_daily_abs_sum"].fillna(0.0).astype(float),
    )
]

print("[§1] nan_reason_w counts across all weekly rows:")
print(weekly["nan_reason_w"].value_counts())

# ── Overlap window: 2024-Q3 → 2025-Q4, six quarter-end anchors ────────────
OVERLAP_QUARTER_ENDS = pd.to_datetime([
    "2024-09-30",  # 2024-Q3
    "2024-12-31",  # 2024-Q4
    "2025-03-31",  # 2025-Q1
    "2025-06-30",  # 2025-Q2
    "2025-09-30",  # 2025-Q3
    "2025-12-31",  # 2025-Q4
])
N_OVERLAP_QUARTERS = len(OVERLAP_QUARTER_ENDS)
K_TRANSITIONS = N_OVERLAP_QUARTERS - 1
assert N_OVERLAP_QUARTERS == 6, (
    f"Overlap N must be 6 (direct enumeration of BanRep quarter-ends in "
    f"2024-Q3..2025-Q4); got {N_OVERLAP_QUARTERS}."
)
print(f"[§1] Overlap window: N = {N_OVERLAP_QUARTERS} quarters, "
      f"K = {K_TRANSITIONS} transitions.")

# ── Per-quarter alignment summary ─────────────────────────────────────────
# For each overlap quarter-end, count how many weekly rows fall in that
# calendar quarter and how they split by nan_reason_w. A weekly row is
# assigned to quarter Q if the row's Friday label lies in [Q_start, Q_end].
weekly["_quarter_end"] = (
    weekly.index.to_period("Q-DEC").to_timestamp(how="end").floor("D").normalize()
    + pd.Timedelta(hours=23, minutes=59, seconds=59)
).normalize()
# Use pandas `resample`/`to_period` semantics to assign each Friday to a
# quarter-end. We normalize to midnight of the last day of the quarter.
weekly["_quarter_end"] = pd.PeriodIndex(weekly.index, freq="Q-DEC").to_timestamp(how="end").normalize()

alignment_rows = []
for qend in OVERLAP_QUARTER_ENDS:
    mask = weekly["_quarter_end"] == qend.normalize()
    sub = weekly.loc[mask]
    alignment_rows.append({
        "quarter_end": qend.date().isoformat(),
        "on_chain_weekly_rows_count": int(mask.sum()),
        "on_chain_dropped_partial_count": int(
            (sub["nan_reason_w"] == "partial_week").sum()
        ),
        "on_chain_zero_valid_count": int(
            (sub["nan_reason_w"] == "all_zero_full_week").sum()
        ),
        "on_chain_valid_count": int((sub["nan_reason_w"] == "valid").sum()),
    })
alignment_df = pd.DataFrame(alignment_rows)
print("[§1] Per-quarter alignment summary:")
print(alignment_df.to_string(index=False))

[§1] Loaded daily CSV: 585 rows (2024-09-17 → 2026-04-24)
[§1] Loaded BanRep quarterly CSV: 104 rows (2000-03-31 → 2025-12-31)
[§1] Task-11.B weekly vector: 84 rows (2024-09-20 → 2026-04-24)
[§1] Weekly vector columns: ['flow_sum_w', 'flow_var_w', 'flow_concentration_w', 'flow_directional_asymmetry_w', 'unique_daily_active_senders_w', 'flow_max_single_day_w']
[§1] nan_reason_w counts across all weekly rows:
nan_reason_w
valid                 81
all_zero_full_week     2
partial_week           1
Name: count, dtype: int64
[§1] Overlap window: N = 6 quarters, K = 5 transitions.
[§1] Per-quarter alignment summary:
quarter_end  on_chain_weekly_rows_count  on_chain_dropped_partial_count  on_chain_zero_valid_count  on_chain_valid_count
 2024-09-30                           2                               1                          1                     0
 2024-12-31                          13                               0                          1                    12
 2025-03-31         

### §1 — Interpretation

**Observed shape of the alignment.** The Task-11.A CSV has 585 daily rows (2024-09-17 → 2026-04-24). The Task-11.B aggregator emits 84 weekly rows (2024-09-20 → 2026-04-24). Across all 84 weekly rows, the `nan_reason_w` partition is: **81 valid, 2 all_zero_full_week, 1 partial_week**. The single partial-week row is the 2024-09-20 Friday that contains only 4 of 7 days (Tue-17 through Fri-20, the first four days of the Task-11.A fixture) — correctly classified as `partial_week` and will be dropped at §2. The two all-zero-full-week rows are 7-day weeks with zero USD flow on every day (pre-cCOP-launch COPM-quiet periods); they are correctly retained as valid-zero.

**Overlap-window shape.** The six overlap quarters split as:

| Quarter | weekly-rows | partial (drop) | all-zero (keep@0) | valid |
|---|---:|---:|---:|---:|
| 2024-Q3 | 2 | 1 | 1 | 0 |
| 2024-Q4 | 13 | 0 | 1 | 12 |
| 2025-Q1 | 13 | 0 | 0 | 13 |
| 2025-Q2 | 13 | 0 | 0 | 13 |
| 2025-Q3 | 13 | 0 | 0 | 13 |
| 2025-Q4 | 13 | 0 | 0 | 13 |

Two observations warrant note. (a) 2024-Q3 contributes very thin data: after dropping the single partial-week row, only one all-zero-full-week row remains — the on-chain quarterly sum for 2024-Q3 will be exactly 0 USD. This is a genuine economic signal (COPM launched 2024-04, cCOP launched 2024-10-31; pre-cCOP-launch activity was small and concentrated in the first half of 2024-Q3). (b) 2024-Q4 contains one all-zero-full-week row inside a quarter otherwise dominated by the cCOP launch — this quarter's sum is dominated by the 12 valid weeks and the single zero-week contributes 0 (adding nothing, dropping nothing). The remaining four quarters (2025-Q1 through 2025-Q4) are full 13-week quarters with zero dropped and zero all-zero rows — clean quarterly sums.

**Risk flag for §3.** 2024-Q3's on-chain sum is mechanically pinned at exactly zero by the pre-launch regime. This is NOT a noise realization — it is a structural feature of the COPM/cCOP launch timeline. A Pearson ρ computed against a series with one mechanically-zero endpoint is sensitive to that endpoint; §3 will report both the full-N=6 ρ and a robustness check excluding 2024-Q3 to confirm the ρ magnitude is not an artifact of the single pre-launch zero.

**HALT — Trio 1 complete.** Awaiting foreground acknowledgement before proceeding to Trio 2 (§2 quarterly aggregation).

## §2 — Quarterly aggregation

**Reference.** Rev-3.4 NaN-ambiguity rule (plan line 329; see also the preamble header of this notebook). Task-11.B emits `flow_sum_w` as a weekly USD-scalar that is already comparable across weeks; the only transformation required at this step is a per-quarter sum with NaN-reason-aware masking.

**Why this step is used.** The BanRep series is intrinsically quarterly; the on-chain series is weekly. Any correlation computation requires one observation per time unit per series, so we sum weekly `flow_sum_w` over the weeks that fall in each overlap quarter, dropping `partial_week` rows (Rev-3.4 rule) and treating `all_zero_full_week` rows as valid zeros (which trivially contribute 0 to the sum — so at the quarterly-sum level the treatment is the same as inclusion, but the distinction matters for the `on_chain_valid_weeks_count` audit column). The joined output has exactly six rows (one per overlap quarter) with two numeric columns: `on_chain_quarterly_sum_usd` and `banrep_quarterly_inflow_usd`.

**Relevance to results.** The quarterly sum is the scalar the Pearson ρ is computed on in §3. It is sensitive to: (a) whether large-flow days are compensated by opposite-sign large-flow days within the same week (the `flow_sum_w` channel is already net of daily sign via the Task-11.B `net_flow = mint+burn+inflow+outflow` — this is flow volume, not signed direction, so both mints and burns contribute positively). (b) whether the pre-launch zero quarter (2024-Q3, see §1 interpretation) mechanically pulls ρ higher by providing a single low-low-corner point — §3 addresses this with a 5-observation robustness check.

**Connection to the primary regression.** The quarterly sum is NOT an input to the primary regression. It exists solely to compute the bridge-ρ. The primary regression uses the full weekly 6-channel vector (`flow_sum_w` plus five intra-week statistics), and the quarterly sum discards the five intra-week statistics by design — quarterly rollup averages out exactly the intra-week information the Rev-3 primary X is designed to preserve. So a strong quarterly-level ρ validates the aggregate-flow channel; a weak one does not invalidate the primary regression, only shifts the economic narrative.

In [2]:
"""§2 — Quarterly aggregation with Rev-3.4 NaN-ambiguity rule."""
from __future__ import annotations

# ── Aggregate weekly flow_sum_w to quarterly USD sums ─────────────────────
# Drop rows with nan_reason_w == "partial_week" (no-data).
# Keep rows with nan_reason_w in {"valid", "all_zero_full_week"}. The
# all-zero rows have flow_sum_w that should be 0.0 after Task-11.B's NaN
# mapping (partial_weeks are NaN'd; all-zero-full-weeks have flow_sum_w = 0
# since the sum of zeros over 7 rows is 0). We coerce any residual NaN in
# non-partial rows to 0.0 to keep the sum well-defined.
_rows_to_keep = weekly.loc[weekly["nan_reason_w"] != "partial_week"].copy()
_rows_to_keep["flow_sum_w_usd"] = _rows_to_keep["flow_sum_w"].fillna(0.0)

# Per-quarter USD sum using the _quarter_end assigned in §1.
quarterly_on_chain = (
    _rows_to_keep.groupby("_quarter_end", sort=True)["flow_sum_w_usd"]
    .sum()
    .rename("on_chain_quarterly_sum_usd")
)

# Audit column: count of weeks contributing (excluding partial_week).
quarterly_on_chain_count = (
    _rows_to_keep.groupby("_quarter_end", sort=True)["flow_sum_w_usd"]
    .size()
    .rename("on_chain_valid_weeks_count")
)

# Restrict to the six overlap quarters.
overlap_index = pd.DatetimeIndex(OVERLAP_QUARTER_ENDS).normalize()
quarterly_on_chain = quarterly_on_chain.reindex(overlap_index, fill_value=0.0)
quarterly_on_chain_count = quarterly_on_chain_count.reindex(overlap_index, fill_value=0)

# BanRep quarterly series in the overlap window. BanRep's `value_usd`
# column is documented as USD MILLIONS (see the CSV's header comment
# "USD millions"). Convert to USD for unit parity with the on-chain sums.
banrep_restricted = banrep.loc[
    banrep["reference_period"].isin(overlap_index)
].sort_values("reference_period")
banrep_q = (
    banrep_restricted.set_index("reference_period")["value_usd"]
    * 1_000_000.0
).rename("banrep_quarterly_inflow_usd")
banrep_q = banrep_q.reindex(overlap_index)

# Join into a single quarterly DataFrame.
quarterly = pd.concat(
    [quarterly_on_chain, banrep_q, quarterly_on_chain_count],
    axis=1,
)
quarterly.index.name = "quarter_end"

print("[§2] Quarterly alignment table (both series in USD, not USD millions):")
# Print with a readable float format.
with pd.option_context(
    "display.float_format",
    lambda v: f"{v:,.2f}",
):
    print(quarterly.to_string())

# Sanity checks.
assert len(quarterly) == N_OVERLAP_QUARTERS, (
    f"Quarterly join produced {len(quarterly)} rows; expected "
    f"{N_OVERLAP_QUARTERS}."
)
assert quarterly["banrep_quarterly_inflow_usd"].notna().all(), (
    "BanRep series has a NaN in the overlap window — cannot proceed."
)
assert quarterly["on_chain_quarterly_sum_usd"].notna().all(), (
    "On-chain quarterly sum has a NaN — NaN-reason masking failed."
)

# Flag quarters with zero on-chain weeks after aggregation (risk flag for §3).
zero_week_quarters = quarterly.loc[
    quarterly["on_chain_valid_weeks_count"] == 0
].index.tolist()
if zero_week_quarters:
    print(f"[§2] RISK-FLAG: quarters with zero valid on-chain weeks "
          f"(mechanical zero contribution): "
          f"{[q.date().isoformat() for q in zero_week_quarters]}")
else:
    print("[§2] All overlap quarters have at least one valid on-chain week.")

[§2] Quarterly alignment table (both series in USD, not USD millions):
             on_chain_quarterly_sum_usd  banrep_quarterly_inflow_usd  on_chain_valid_weeks_count
quarter_end                                                                                     
2024-09-30                         0.00             3,052,700,000.00                           1
2024-12-31                   898,766.34             3,167,990,000.00                          13
2025-03-31                 7,069,193.29             3,130,620,000.00                          13
2025-06-30                 3,697,897.06             3,277,470,000.00                          13
2025-09-30                16,681,562.69             3,354,000,000.00                          13
2025-12-31                28,691,059.42             3,336,180,000.00                          13
[§2] All overlap quarters have at least one valid on-chain week.


### §2 — Interpretation

**Observed quarterly table.**

| Quarter | On-chain sum (USD) | BanRep inflow (USD) | Valid on-chain weeks |
|---|---:|---:|---:|
| 2024-Q3 | 0 | 3,052,700,000 | 1 |
| 2024-Q4 | 898,766 | 3,167,990,000 | 13 |
| 2025-Q1 | 7,069,193 | 3,130,620,000 | 13 |
| 2025-Q2 | 3,697,897 | 3,277,470,000 | 13 |
| 2025-Q3 | 16,681,563 | 3,354,000,000 | 13 |
| 2025-Q4 | 28,691,059 | 3,336,180,000 | 13 |

**Scale observation (crucial for interpretation).** The BanRep aggregate remittance inflows are ~$3 billion per quarter. The on-chain aggregate tops out at ~$29 million in 2025-Q4. At 2025-Q4 the on-chain rail is carrying approximately **0.86% of the BanRep aggregate** — i.e. the on-chain COPM+cCOP channel is a small-but-growing slice of the total Colombian-diaspora remittance corridor, not a macro-scale proxy. This is consistent with the product-framing in `REMITTANCE_VOLATILITY_SWAP.md` (cCOP is one of several rails; the majority of Colombian remittance still routes through traditional wire services).

**Growth-trajectory observation.** The on-chain quarterly sum shows clear growth with one non-monotone step: 0 → 898k → 7.07M → 3.70M → 16.68M → 28.69M. The Q2 step-down (7.07M → 3.70M) interrupts an otherwise monotonically increasing series. BanRep, in contrast, is seasonally flat: 3.05 → 3.17 → 3.13 → 3.28 → 3.35 → 3.34B. The two series' Δ patterns should determine whether sign-concordance is met.

**Implications for §3 Pearson ρ.**
- If ρ is driven by the 2024-Q3 corner zero (on-chain = 0 USD paired with BanRep = $3.05B, the lowest BanRep quarter in the overlap window), we expect the ρ to be inflated. §3 will compute both the full-6-observation ρ and a robustness 5-observation ρ excluding 2024-Q3.
- The BanRep series is near-constant relative to its own level (std-over-mean ≈ 3.6%) while the on-chain series ranges over 3 orders of magnitude. Pearson ρ is scale-invariant on centered-and-normalized series, so the magnitude difference does not bias ρ per se, but it does mean the BanRep variance component of the ρ denominator is tiny — ρ is mostly driven by whether the on-chain ranks match BanRep's small ranks in sign.

**HALT — Trio 2 complete.** Awaiting foreground acknowledgement before proceeding to Trio 3 (§3 Pearson ρ + sign-concordance).

## §3 — Pearson ρ and sign-concordance

**Reference.** `scipy.stats.pearsonr` for the Pearson product-moment correlation (two-series, equal-length, small-sample). Sign-concordance is a direct enumeration of Δ Q-over-Q transitions.

**Why this step is used.** Per the Rev-3 plan line 317-319, the bridge gate is defined on TWO criteria (Pearson ρ magnitude AND sign-concordance on quarter-over-quarter differences). Both must be computed before §4's verdict emission. The sign-concordance criterion guards against a level-correlation that is driven by common trend rather than by genuine mechanism-linked co-movement — if the on-chain series and BanRep both happen to trend upward over the sample but their quarter-to-quarter deltas do not align, the ρ is spurious in the sense that it would not survive detrending.

**Relevance to results.** The pair (ρ, concordance-count) fully determines the pre-registered gate verdict. With N=6 and K=5:
- PASS-BRIDGE requires ρ > 0.5 AND concordance-count ≥ 3.
- INCONCLUSIVE-BRIDGE requires 0.3 < ρ ≤ 0.5 AND concordance-count ≥ 3.
- FAIL-BRIDGE triggers if ρ ≤ 0.3 OR concordance-count < 3.

We also compute a 5-observation robustness ρ (dropping the 2024-Q3 mechanical-zero corner) and the associated concordance on 4 transitions. The robustness ρ is NOT a second gate — it is reported for transparency but the primary verdict uses the 6-observation ρ per pre-commitment. Adjusting the gate based on the robustness ρ would be post-hoc fishing.

**Connection to the primary regression.** The primary regression's interpretation depends on the bridge verdict: PASS-BRIDGE → "on-chain aggregate proxies BanRep remittance inflow, so regression β̂_Rem is a remittance-channel coefficient"; FAIL-BRIDGE → "on-chain aggregate is a crypto-rail income-conversion signal distinct from aggregate remittance, so regression β̂_Rem is a crypto-rail coefficient"; INCONCLUSIVE → primary regression runs; narrative retains the remittance label with a documented caveat.

In [3]:
"""§3 — Pearson ρ and sign-concordance (primary + robustness)."""
from __future__ import annotations

from scipy import stats


# ── Primary Pearson ρ on the full N=6 quarterly sample ────────────────────
on_chain_series = quarterly["on_chain_quarterly_sum_usd"].to_numpy(dtype="float64")
banrep_series = quarterly["banrep_quarterly_inflow_usd"].to_numpy(dtype="float64")

pearson_result_primary = stats.pearsonr(on_chain_series, banrep_series)
rho_primary = float(pearson_result_primary.statistic)
pval_primary = float(pearson_result_primary.pvalue)

print(f"[§3] Primary Pearson ρ (N=6): {rho_primary:+.6f}   "
      f"(two-sided p-value = {pval_primary:.4f})")

# ── Robustness Pearson ρ excluding 2024-Q3 (N=5) ──────────────────────────
mask_excl_q3 = quarterly.index != pd.Timestamp("2024-09-30")
on_chain_r = quarterly.loc[mask_excl_q3, "on_chain_quarterly_sum_usd"].to_numpy(
    dtype="float64"
)
banrep_r = quarterly.loc[mask_excl_q3, "banrep_quarterly_inflow_usd"].to_numpy(
    dtype="float64"
)
pearson_result_robust = stats.pearsonr(on_chain_r, banrep_r)
rho_robust = float(pearson_result_robust.statistic)
pval_robust = float(pearson_result_robust.pvalue)

print(f"[§3] Robustness Pearson ρ (N=5, exc. 2024-Q3): "
      f"{rho_robust:+.6f}   (two-sided p-value = {pval_robust:.4f})")

# ── Sign-concordance on Δ Q-over-Q (K=5 transitions) ──────────────────────
# For each of the 5 transitions, does Δ(on-chain) share sign with Δ(BanRep)?
delta_on_chain = np.diff(on_chain_series)
delta_banrep = np.diff(banrep_series)
assert len(delta_on_chain) == K_TRANSITIONS, "Δ count mismatch"
assert len(delta_banrep) == K_TRANSITIONS, "Δ count mismatch"

sign_on_chain = np.sign(delta_on_chain)
sign_banrep = np.sign(delta_banrep)

concordance_flags = (sign_on_chain == sign_banrep) & (sign_on_chain != 0)
concordance_count = int(concordance_flags.sum())

print(f"[§3] Δ Q-over-Q (K={K_TRANSITIONS} transitions):")
_q_dates = [q.date().isoformat() for q in quarterly.index]
for k in range(K_TRANSITIONS):
    print(
        f"       {_q_dates[k]} → {_q_dates[k+1]}: "
        f"Δ_on-chain = {delta_on_chain[k]:+,.2f}  "
        f"Δ_BanRep = {delta_banrep[k]:+,.2f}  "
        f"concordant = {bool(concordance_flags[k])}"
    )
print(f"[§3] Sign-concordance count = {concordance_count}/{K_TRANSITIONS}")

# ── Pre-registered-gate boolean components (no verdict yet — that is §4) ──
SIGN_CONCORDANCE_MINIMUM = 3  # ≥3 of 5 transitions (committed in preamble)
is_sign_concordant = concordance_count >= SIGN_CONCORDANCE_MINIMUM
is_rho_above_pass = rho_primary > 0.5
is_rho_in_inconclusive_band = (rho_primary > 0.3) and (rho_primary <= 0.5)
is_rho_at_or_below_fail = rho_primary <= 0.3

print(f"[§3] Gate-input booleans (applied in §4 WITHOUT re-reading rho):")
print(f"       sign_concordance_count >= {SIGN_CONCORDANCE_MINIMUM}: "
      f"{is_sign_concordant}")
print(f"       rho_primary > 0.5: {is_rho_above_pass}")
print(f"       0.3 < rho_primary <= 0.5: {is_rho_in_inconclusive_band}")
print(f"       rho_primary <= 0.3: {is_rho_at_or_below_fail}")

[§3] Primary Pearson ρ (N=6): +0.755417   (two-sided p-value = 0.0824)
[§3] Robustness Pearson ρ (N=5, exc. 2024-Q3): +0.706886   (two-sided p-value = 0.1819)
[§3] Δ Q-over-Q (K=5 transitions):
       2024-09-30 → 2024-12-31: Δ_on-chain = +898,766.34  Δ_BanRep = +115,290,000.00  concordant = True
       2024-12-31 → 2025-03-31: Δ_on-chain = +6,170,426.96  Δ_BanRep = -37,370,000.00  concordant = False
       2025-03-31 → 2025-06-30: Δ_on-chain = -3,371,296.24  Δ_BanRep = +146,850,000.00  concordant = False
       2025-06-30 → 2025-09-30: Δ_on-chain = +12,983,665.63  Δ_BanRep = +76,530,000.00  concordant = True
       2025-09-30 → 2025-12-31: Δ_on-chain = +12,009,496.73  Δ_BanRep = -17,820,000.00  concordant = False
[§3] Sign-concordance count = 2/5
[§3] Gate-input booleans (applied in §4 WITHOUT re-reading rho):
       sign_concordance_count >= 3: False
       rho_primary > 0.5: True
       0.3 < rho_primary <= 0.5: False
       rho_primary <= 0.3: False


### §3 — Interpretation

**Observed results.**
- **Primary Pearson ρ (N=6): +0.7554** (two-sided p = 0.0824)
- **Robustness Pearson ρ (N=5, excluding 2024-Q3): +0.7069** (two-sided p = 0.1819)
- **Sign-concordance on Δ Q-over-Q: 2 of 5**

**Level correlation is positive and sizeable.** ρ = +0.755 on N=6 is a large Pearson magnitude, and the robustness ρ of +0.707 (dropping the mechanical-zero 2024-Q3 corner) is nearly identical — so the level-correlation is NOT an artifact of the pre-launch zero. Both series trended upward over the six-quarter window and that joint trend produces the ρ.

**But the quarter-over-quarter deltas do NOT align.** Only 2 of 5 transitions are concordant:
- 2024-Q3 → Q4: both up → **concordant**
- 2024-Q4 → 2025-Q1: on-chain UP, BanRep DOWN → **discordant**
- 2025-Q1 → Q2: on-chain DOWN, BanRep UP → **discordant**
- 2025-Q2 → Q3: both up → **concordant**
- 2025-Q3 → Q4: on-chain UP, BanRep DOWN → **discordant**

The sign-concordance count of 2/5 is BELOW the pre-registered threshold of 3/5, so the sign-concordance criterion fails. Combined with the FAIL-BRIDGE gate's OR clause ("ρ ≤ 0.3 OR sign-discordant"), the pre-registered gate emits **FAIL-BRIDGE** in §4 — the level-correlation is driven by common trend (both series trending up over the sample), not by mechanism-linked quarter-to-quarter co-movement.

**Economic reading of the discordance.** The on-chain series is in its adoption-ramp phase (Apr-2024 COPM launch, Oct-2024 cCOP launch, mostly non-stationary growth), while BanRep aggregate inflows are near-constant around $3.0-3.4B per quarter (Colombian diaspora flows are large, slow-moving, and relatively insensitive to a single-quarter crypto-rail feature). The on-chain series' quarter-to-quarter swings reflect cCOP/COPM-specific adoption dynamics; the BanRep series' quarter-to-quarter swings reflect macro-remittance seasonality (holiday effects, exchange-rate level, diaspora employment). These are plausibly distinct economic phenomena — a FAIL-BRIDGE is the expected outcome under the hypothesis that the on-chain rail is a minority-share channel with its own dynamics, NOT a proxy for the aggregate remittance corridor.

**Pre-registration discipline reminder.** The observed ρ of +0.755 exceeds the PASS-BRIDGE threshold of 0.5. A less-disciplined author could at this point drop the sign-concordance criterion and declare PASS-BRIDGE — but that would be post-hoc fishing exactly of the type the Rev-3 gate was designed to prevent. The AND clause in the PASS-BRIDGE rule is load-bearing, and §4 will honour it.

**HALT — Trio 3 complete.** Awaiting foreground acknowledgement before proceeding to Trio 4 (§4 verdict emission).

## §4 — Verdict emission

**Reference.** Rev-3 plan line 316-319 (pre-registered gate) + Rev-3.1 recovery protocol (plan line 324-328) + Rev-3.4 NaN-ambiguity rule (plan line 329) + this notebook's preamble cell (committed BEFORE any ρ was computed).

**Why this step is used.** §4 is the final step of the bridge-validation protocol: apply the pre-registered gate logic to the observed ρ + sign-concordance values (from §3), classify the outcome into one of three enum verdicts, and persist that verdict to the scratch-log artifact `contracts/.scratch/2026-04-20-onchain-banrep-bridge-result.md`. The scratch log is the single external-memory artifact for this task (no JSON/pickle; notebook 0B is one-off validation) and is the object Task 30a's completion memory will later reference.

**Relevance to results.** This cell encodes the pre-committed gate decisions exactly as they appeared in the preamble, without consulting the observed ρ in the code. The gate-boolean inputs (`is_sign_concordant`, `is_rho_above_pass`, `is_rho_in_inconclusive_band`, `is_rho_at_or_below_fail`) were prepared in §3 so the verdict-classification code in this cell can be read and audited as a pure logic function of those booleans.

**Connection to the primary regression.** Per Rev-3.1 recovery protocol:
- **FAIL-BRIDGE:** primary regression still runs; narrative shifts from "remittance" to "crypto-rail income-conversion"; `Rev-1.1.1 spec patch` re-scopes §4.1 only at the time FAIL is observed (deferred to Task 11.D implementation time).
- **INCONCLUSIVE-BRIDGE:** primary regression still runs; caveat documented; no spec patch.
- **PASS-BRIDGE:** primary regression proceeds; no narrative shift.

Gate thresholds are NOT adjusted based on observed values (the CPI-surprise exercise's core anti-fishing lesson, carried into Phase-A.0).

In [4]:
"""§4 — Pre-registered-gate verdict classification + scratch-log emission."""
from __future__ import annotations

import datetime as _dt


# ── Pure classifier: applies the pre-committed gate without reading rho ───
def classify_bridge_verdict(
    *,
    rho_above_pass: bool,
    rho_in_inconclusive_band: bool,
    rho_at_or_below_fail: bool,
    sign_concordant: bool,
) -> str:
    """Return one of {'PASS-BRIDGE', 'INCONCLUSIVE-BRIDGE', 'FAIL-BRIDGE'}.

    Gate logic (committed in the preamble, line-for-line):
      * PASS-BRIDGE if (ρ > 0.5) AND sign-concordant.
      * FAIL-BRIDGE if (ρ ≤ 0.3) OR NOT sign-concordant.
      * INCONCLUSIVE-BRIDGE otherwise (0.3 < ρ ≤ 0.5 AND sign-concordant).
    """
    # FAIL-BRIDGE takes precedence over the other two (the OR clause catches
    # both "low ρ" and "high ρ but sign-discordant" cases).
    if rho_at_or_below_fail or (not sign_concordant):
        return "FAIL-BRIDGE"
    if rho_above_pass and sign_concordant:
        return "PASS-BRIDGE"
    if rho_in_inconclusive_band and sign_concordant:
        return "INCONCLUSIVE-BRIDGE"
    # Defensive: the three cases above should be exhaustive.
    raise RuntimeError(
        "bridge verdict classifier reached an unreachable branch — "
        "gate logic inputs were not mutually exhaustive."
    )


verdict = classify_bridge_verdict(
    rho_above_pass=is_rho_above_pass,
    rho_in_inconclusive_band=is_rho_in_inconclusive_band,
    rho_at_or_below_fail=is_rho_at_or_below_fail,
    sign_concordant=is_sign_concordant,
)

print(f"[§4] Pre-registered gate verdict: {verdict}")

# Narrative-implication string per Rev-3.1 recovery protocol.
if verdict == "PASS-BRIDGE":
    narrative = (
        "Primary regression proceeds without narrative shift. The on-chain "
        "aggregate is validated as a proxy for the BanRep aggregate remittance "
        "channel; β̂_Rem retains the 'remittance' label."
    )
elif verdict == "FAIL-BRIDGE":
    narrative = (
        "Primary regression still runs (the on-chain X is a well-defined "
        "observable). Narrative shifts from 'remittance' to 'crypto-rail "
        "income-conversion' per Rev-3.1 recovery protocol item 1. A Rev-1.1.1 "
        "spec patch (to be authored at FAIL-BRIDGE observation) re-scopes §4.1 "
        "only; §§4.2+ mechanism unchanged. Task 11.D decision gate: the spec "
        "patch is classified as wording/cadence-only (X-interpretation relabel), "
        "not an economic-mechanism change."
    )
else:  # INCONCLUSIVE-BRIDGE
    narrative = (
        "Primary regression still runs; completion memory documents the "
        "inconclusive bridge as a caveat. No spec patch required."
    )

print(f"[§4] Narrative-implication: {narrative}")

# ── Emit scratch-log artifact ────────────────────────────────────────────
SCRATCH_LOG_PATH = (
    WORKTREE_ROOT / "contracts" / ".scratch" / "2026-04-20-onchain-banrep-bridge-result.md"
)
SCRATCH_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)

_now_iso = _dt.datetime.now(_dt.timezone.utc).isoformat(timespec="seconds")

_log_body = f"""# Task 11.C Bridge-Validation Result (Phase-A.0 Rev-3.4)

**Verdict (pre-registered gate):** `{verdict}`

**Execution timestamp (UTC):** {_now_iso}

## Observed bridge statistics

| Statistic | Value |
|---|---|
| Pearson ρ (N=6 primary) | `{rho_primary:+.6f}` (two-sided p = `{pval_primary:.4f}`) |
| Pearson ρ (N=5 robustness, excl. 2024-Q3) | `{rho_robust:+.6f}` (two-sided p = `{pval_robust:.4f}`) |
| Sign-concordance count | `{concordance_count}/{K_TRANSITIONS}` (threshold ≥ {SIGN_CONCORDANCE_MINIMUM}) |
| Sign-concordant? | `{is_sign_concordant}` |
| ρ > 0.5? | `{is_rho_above_pass}` |
| 0.3 < ρ ≤ 0.5? | `{is_rho_in_inconclusive_band}` |
| ρ ≤ 0.3? | `{is_rho_at_or_below_fail}` |

## Overlap window

Observed N = 6 quarterly observations (task description prescribed N = 7, off
by one; see preamble header of `0B_bridge_validation.ipynb`). K = 5 Q-over-Q
transitions. Quarterly sample: 2024-Q3, 2024-Q4, 2025-Q1, 2025-Q2, 2025-Q3,
2025-Q4.

## Pre-registered gate (committed BEFORE any ρ computation)

| Verdict | Condition |
|---|---|
| PASS-BRIDGE | ρ > 0.5 AND sign-concordant (≥ 3 of 5 transitions agree) |
| INCONCLUSIVE-BRIDGE | 0.3 < ρ ≤ 0.5 AND sign-concordant |
| FAIL-BRIDGE | ρ ≤ 0.3 OR sign-discordant (< 3 of 5 transitions agree) |

## Narrative implication (Rev-3.1 recovery protocol)

{narrative}

## Rev-3.4 NaN-ambiguity protocol applied

Quarterly aggregation distinguished `partial_week` (dropped) from
`all_zero_full_week` (retained as valid zero) per plan line 329. Across the
84 weekly rows emitted by Task-11.B: 1 partial_week, 2 all_zero_full_week,
81 valid. The single partial_week (2024-09-20 boundary Friday) was dropped;
the two all_zero_full_week rows contributed 0 USD to their respective
quarterly sums.

## Source artifact paths

- Task-11.A daily CSV: `contracts/data/copm_ccop_daily_flow.csv`
- Task-11.B aggregator: `contracts/scripts/weekly_onchain_flow_vector.py`
- Task-11 BanRep quarterly CSV: `contracts/data/banrep_remittance_aggregate_monthly.csv`
- This notebook: `contracts/notebooks/fx_vol_remittance_surprise/Colombia/0B_bridge_validation.ipynb`

## Quarterly bridge table (USD)

```
{quarterly.to_string(float_format=lambda v: f'{v:,.2f}')}
```

## Δ Q-over-Q transitions

| Transition | Δ on-chain (USD) | Δ BanRep (USD) | Concordant? |
|---|---:|---:|---:|
"""
for k in range(K_TRANSITIONS):
    _log_body += (
        f"| {_q_dates[k]} → {_q_dates[k+1]} | "
        f"{delta_on_chain[k]:+,.2f} | "
        f"{delta_banrep[k]:+,.2f} | "
        f"{bool(concordance_flags[k])} |\n"
    )

_log_body += f"""
## Anti-fishing framing (Rev-1 §10)

This exercise is a distinct pre-commitment on the remittance external-inflow
channel via the on-chain COPM+cCOP rail. It is NOT a rescue of the CPI-FAIL
verdict (2026-04-19). Provenance anchor:
`/home/jmsbpp/apps/liq-soldk-dev/notes/REMITTANCE_VOLATILITY_SWAP/REMITTANCE_VOLATILITY_SWAP.md`
has mtime 2026-04-02 (17 days before CPI-FAIL).

The pre-registered gate thresholds were NOT adjusted after observing the ρ
value of {rho_primary:+.4f}. The sign-concordance AND-clause is load-bearing in the
PASS-BRIDGE rule and was honored: although ρ > 0.5, the 2/5 concordance
count triggered the FAIL-BRIDGE OR-clause.
"""

SCRATCH_LOG_PATH.write_text(_log_body, encoding="utf-8")
print(f"[§4] Scratch log written: {SCRATCH_LOG_PATH}")
print(f"[§4] Log byte-size: {SCRATCH_LOG_PATH.stat().st_size}")

[§4] Pre-registered gate verdict: FAIL-BRIDGE
[§4] Narrative-implication: Primary regression still runs (the on-chain X is a well-defined observable). Narrative shifts from 'remittance' to 'crypto-rail income-conversion' per Rev-3.1 recovery protocol item 1. A Rev-1.1.1 spec patch (to be authored at FAIL-BRIDGE observation) re-scopes §4.1 only; §§4.2+ mechanism unchanged. Task 11.D decision gate: the spec patch is classified as wording/cadence-only (X-interpretation relabel), not an economic-mechanism change.
[§4] Scratch log written: /home/jmsbpp/apps/ThetaSwap/thetaSwap-core-dev/.worktree/ranFromAngstrom/contracts/.scratch/2026-04-20-onchain-banrep-bridge-result.md
[§4] Log byte-size: 4336


### §4 — Interpretation

**Emitted verdict: `FAIL-BRIDGE`.**

**Gate arithmetic.** The classifier evaluated the four pre-committed booleans:
- `rho_above_pass` = True (ρ = +0.7554 > 0.5)
- `rho_in_inconclusive_band` = False
- `rho_at_or_below_fail` = False
- `sign_concordant` = False (concordance-count 2 < 3 threshold)

FAIL-BRIDGE condition: `rho_at_or_below_fail OR (not sign_concordant)` → `False OR True` → **True** → FAIL-BRIDGE.

**Narrative shift (Rev-3.1 recovery protocol item 1).** The primary Phase-A.0 regression still runs (the on-chain X is a well-defined observable). The economic narrative shifts from "remittance" to "crypto-rail income-conversion": the on-chain COPM+cCOP aggregate is not validated as a statistically-meaningful proxy for the BanRep aggregate remittance channel, but it IS a well-defined, directly-observed crypto-rail flow series. Task 11.D (Rev-1.1 spec patch) will re-scope §4.1's X-interpretation at authoring time to reflect this; §§4.2+ (mechanism, identification, estimation, tests) are untouched because the X-measurement is unchanged — only its economic label is.

**Abrigo product read.** At 2025-Q4 the on-chain rail is carrying ~0.86% of the BanRep aggregate. The FAIL-BRIDGE result is economically consistent with this scale: a sub-1%-share channel is unlikely to correlate quarter-to-quarter with a macro-aggregate dominated by the other 99% (wire services, informal channels, and non-crypto fintech rails). For Abrigo's painkiller positioning, the FAIL-BRIDGE reading is honest but NOT fatal — it means the Phase-A.0 result, whatever it turns out to be for β̂_Rem, speaks to crypto-rail-user volatility-hedging demand (a natural early-adopter audience for Abrigo) rather than to aggregate-diaspora remittance hedging. The distinction is product-relevant: the early Abrigo customer is the crypto-rail user, not the aggregate remittance sender, and a crypto-rail-specific signal is the more direct input to underwriting the first-wave product.

**Scratch-log artifact.** Written to `contracts/.scratch/2026-04-20-onchain-banrep-bridge-result.md`. The log is the sole external artifact of this notebook and is the object Task 30a (completion memory) will reference as "bridge-validation scratch note from Task 11.C."

**End of notebook 0B.**